In [1]:
# 📈 TradeSim — Launcher Notebook

Run the single cell below to check packages, start the app, and stream logs.

This notebook provides a sequential runner (one cell) so you can run step-by-step or all at once.

Keyboard shortcuts in the app:

- `Space` or `→`: advance +1 bar
- `Shift + →`: advance +5 bars
- `Ctrl + →`: advance +10 bars
- `Alt + →`: advance +50 bars
- `R`: reset to start
- `P`: toggle auto-play


SyntaxError: invalid character '→' (U+2192) (1052440489.py, line 9)

In [3]:
# ── Run: package check → start app → show logs (single-step runner)
import importlib, subprocess, sys, time, os, socket, urllib.request
from pathlib import Path
from IPython.display import display, HTML

APP_DIR  = Path(r"C:\Users\User\Desktop\ml\SOHNIMDR\tradesim")
APP_FILE = APP_DIR / "app.py"
PORTS    = list(range(8501, 8511))

# 1) Package check
required = {
    "streamlit":                    "streamlit",
    "streamlit_lightweight_charts": "streamlit-lightweight-charts",
    "yfinance":                     "yfinance",
    "pandas":                       "pandas",
    "numpy":                        "numpy",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("✅ Packages installed")
else:
    print("✅ All packages present")

# 2) Kill existing streamlit processes (best-effort)
try:
    subprocess.call([
        "powershell", "-Command",
        "Get-Process -Name 'streamlit' -ErrorAction SilentlyContinue | Stop-Process -Force"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception:
    pass

# 3) Find free port
def _port_free(p: int) -> bool:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.bind(("127.0.0.1", p))
        s.close()
        return True
    except Exception:
        return False

PORT = next((p for p in PORTS if _port_free(p)), 8501)

# 4) Start streamlit
print(f"Starting TradeSim on port {PORT}...")
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(APP_FILE),
     "--server.port", str(PORT),
     "--browser.gatherUsageStats", "false",
     "--server.headless", "true"],
    cwd=str(APP_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# 5) Wait for server to accept connections
url = f"http://localhost:{PORT}"
for _ in range(40):
    time.sleep(0.5)
    if proc.poll() is not None:
        out, _ = proc.communicate()
        print("❌ Streamlit crashed:\n", out)
        raise RuntimeError("Streamlit failed to start")
    try:
        urllib.request.urlopen(url, timeout=1)
        break
    except Exception:
        pass
else:
    print("Timed out waiting for server")

print(f"✅ TradeSim running at {url}")

# show launch card
display(HTML(f"""
<div style="background:#1E293B; border:2px solid #3B82F6; border-radius:12px; padding:20px; margin:10px 0; font-family:Inter,system-ui,sans-serif; text-align:center;">
  <div style="color:#94A3B8; font-size:13px; margin-bottom:8px;">TradeSim is ready</div>
  <a href="{url}" target="_blank" style="display:inline-block; background:#3B82F6; color:white; padding:10px 28px; border-radius:8px; font-weight:700; font-size:16px; text-decoration:none;">📈 Open TradeSim →</a>
  <div style="color:#64748B; font-size:11px; margin-top:8px;">{url}</div>
</div>
"""))

# 6) Tail logs in background and expose `proc`
import threading, queue
_log_q: queue.Queue = queue.Queue()

def _tail(p):
    for line in iter(p.stdout.readline, ""):
        _log_q.put(line)

threading.Thread(target=_tail, args=(proc,), daemon=True).start()
get_ipython().user_global_ns['proc'] = proc
print('(Logs are being captured; run the next cell to print recent lines)')


✅ All packages present
Starting TradeSim on port 8501...
✅ TradeSim running at http://localhost:8501


(Logs are being captured; run the next cell to print recent lines)


In [4]:
# ── Cell: show logs (run after launcher)
import time
lines = []
try:
    import queue
    while not _log_q.empty():
        lines.append(_log_q.get_nowait())
except Exception:
    pass

if lines:
    print(''.join(lines))
else:
    print('(no new log lines)')


2026-02-25 12:59:21.980 Port 8501 is not available
2026-02-25 13:02:28.361 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-02-25 13:02:29.236 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-02-25 13:13:18.004 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-02-25 13:13:18.008 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=Fals

In [3]:
# ── Cell: tail logs continuously (optional)
# Run this cell to stream logs in the notebook until you stop it.
import time
try:
    while True:
        while not _log_q.empty():
            print(_log_q.get_nowait(), end='')
        time.sleep(0.5)
except KeyboardInterrupt:
    print('\nStopped log tail')


2026-02-25 12:53:50.638 Script compilation error
Traceback (most recent call last):
  File "c:\Users\User\anaconda3\envs\trade310\lib\site-packages\streamlit\runtime\scriptrunner\script_runner.py", line 591, in _run_script
    code = self._script_cache.get_bytecode(script_path)
  File "c:\Users\User\anaconda3\envs\trade310\lib\site-packages\streamlit\runtime\scriptrunner\script_cache.py", line 72, in get_bytecode
    filebody = magic.add_magic(filebody, script_path)
  File "c:\Users\User\anaconda3\envs\trade310\lib\site-packages\streamlit\runtime\scriptrunner\magic.py", line 45, in add_magic
    tree = ast.parse(code, script_path, "exec")
  File "c:\Users\User\anaconda3\envs\trade310\lib\ast.py", line 50, in parse
    return compile(source, filename, mode, flags,
  File "C:\Users\User\Desktop\ml\SOHNIMDR\tradesim\app.py", line 109
    </style>""", unsafe_allow_html=True)
               ^
SyntaxError: f-string: single '}' is not allowed
2026-02-25 12:53:50.649 Thread 'MainThread': missi

In [ ]:
# ── Cell 4: עצירת האפליקציה ───────────────────────────────────────────────
proc.terminate()
proc.wait()
print("🛑 TradeSim stopped")

🛑 TradeSim stopped
